# 18.3 只能看到教師回答也能學嗎？


如果服務只回傳教師的回答文字，拿不到各候選分數，仍可以用這些文字教學生。做法是把題目與教師回答保存成對話，檢查品質，再照[對話SFT](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)訓練學生，讓學生提高回答中各token的機率。這就是[黑盒蒸餾](https://birdhackor.github.io/tiny-perceptron-vlm/18.1.html)，不是逼學生重建未取得的完整分布。

我們希望學生對「2+2=?」回答4。材料裡卻有兩筆教師回答，一筆4、一筆5。我們知道標準答案是4，應先排除5，而不是期待學生自己發現教師錯誤。下面故意用人工記錄模擬格式，沒有呼叫任何教師；來源欄也清楚標成示例，避免教材跑完後產生「已收集教師資料」的誤解。實際生成應保存教師權重版本、提示、生成設定與檢查結果，才能追蹤錯誤來源。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import json

records = [{"numbers": [2, 2], "answer": "4"}, {"numbers": [2, 2], "answer": "5"}]
accepted = []
for record in records:
    if record["answer"] == str(sum(record["numbers"])):
        accepted.append(
            {
                "messages": [{"role": "user", "content": "2+2=?"}, {"role": "assistant", "content": record["answer"]}],
                "teacher_revision": None,
                "provenance": "人工示例，非教師生成",
            }
        )
print("保留筆數", len(accepted))
for record in accepted:
    print(json.dumps(record, ensure_ascii=False))

輸入兩筆字典，numbers只供這個簡單驗證器核對數學，answer是要交給學生的文字。`sum`算出真值，`str`轉成文字再比較；不合格記錄不進入accepted。輸出應保留一筆，內容包含user問題與assistant的4。`json.dumps`將字典編成JSON，`ensure_ascii=False`讓中文直接顯示；若將每筆JSON各存一行，就稱JSONL。此程式只印出，不建立資料檔也不訓練模型。

學生訓練時應讓assistant回答位置計入誤差，user題目作可見前文，回顧[哪些位置應該學](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html)。回答先作為文字取得，再由學生自己的tokenizer切分，所以教師與學生不必使用同一詞表。這與白盒逐欄機率比較不同，不要把有answer字段當成已經有teacher logits。

對保留下來的這筆題目，如果原資料本來也標「4」，而教師回答切成學生token後，連同EOS結束位置、前文與計入誤差的位置都相同，兩條路就是在教同一份答案。使用同樣的答案誤差與樣本權重時，沒有新增監督訊號；把來源欄改成「教師」不會讓誤差公式多出知識。若初始權重、batch次序、優化器與隨機運算也一致，訓練才會沿著相同路徑進行。

若教師改交「2個加2個共有4個，所以答案是4」，收到的整串示範就不同了。它可能多教了解答過程或風格，也增加要學的token；是否正確、有用，仍須檢查。這串文字仍沒有提供每個位置未被選中的候選比例，不能當成完整分布。

生成本身會花教師算力，資料還需要驗證與保存，成本不能只算學生訓練。對於無法用加法檢查的任務，需依來源、規格或評分規則檢查，不能用學生模仿程度當真值。GSM8K是收錄小學程度文字算術題與人寫解答的公開資料集；若使用其原有答案，也應保存原資料來源，不能換一個provenance名稱就宣稱由教師生成。

練習換成翻譯：原資料把「Good morning」標為「早安」，教師甲也回答「早安」，教師乙回答「早上好」。固定同一個學生切詞器與訓練方式，哪一份會改變完整的文字目標？兩種譯法都合適，是否就表示訓練訊號完全相同？又能否由任一譯文還原教師的整列機率？請分別說明目標文字與語意正確性的判斷，不用啟動訓練。

<details>
<summary>核對你的推理</summary>

若其他目標位置與結束條件相同，教師甲沒有改變原答案；教師乙改變了要學的文字與token序列。兩種譯法意思合適，不代表每一步的目標相同，也不能單憑這點宣布哪條路較好。兩份譯文都沒有保存完整候選機率。

</details>

<details>
<summary>選讀：實際硬回答、錯誤審核與正常結束</summary>

上方驗證器示範篩選；正式實驗的`teacher_hard`對照則保留全部實際教師回答，量出其錯誤後果。屬性教師對45筆訓練題全答對、全部EOS，因此硬目標的原始ID恰好和真值相同；同一初始化與batch計畫下，CE與教師硬目標學生的權重及成績也相同，這不是額外知識帶來的提升。風格教師185筆中180筆正確，另外五筆已知日期卻把日子加10，例如要求確認`2026-10-06`，回答`已確認2026-10-16。`，這些錯標籤仍保存在`style-hard-targets.json`的審核記錄。

本輪兩組共230個硬目標都有實際EOS、無非法控制ID，也沒有零長度目標。本輪教師與學生共用同一套264-ID byte詞表與特殊符號約定，同一ID代表相同token；這是這次實驗能直接沿用ID的前提。工具逐筆保存完整`teacher_ids`與EOS旗標，再直接用原始ID作標籤；若教師達生成上限卻未EOS，就不補一個假的結束。零個目標則明確失敗並保存題數，不能悄悄丟題。對只有文字的外部服務，結束原因也應另外保存，普通文字解碼可能隱藏控制token，不能由「畫面像一句完整話」推定正常停止。

</details>
